# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

Skill used: `hunting-leakage-and-validating` (+ `flyrank/flyrank-data` for the dataset gotchas).
This runs the attack-your-own-features checklist BEFORE any model is trained, on the same
starter file used throughout (`data/raw/content_refresh_anonymized.csv`).


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashkrverma1234-glitch/ml-internship-assignment1"
REPO_DIR = "ml-internship-assignment1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
os.makedirs("work/outputs", exist_ok=True)
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Rows: {len(df):,}  Clients: {df['client_id'].nunique()}  Columns: {df.shape[1]}")
print(f"Base rate (is_declining_label=1): {df['is_declining_label'].mean():.3f}")


Rows: 30,000  Clients: 32  Columns: 45
Base rate (is_declining_label=1): 0.542


## 1. Build the feature vector

Numeric traffic columns are heavy-tailed (per `flyrank-data`'s own warning), so the volume
signals go in as `log1p()` rather than raw counts. Missingness follows `content_type` (the data
dictionary's own gotcha) — rather than a blind `fillna(0)`, missing numeric fields get an
explicit `has_<field>` flag plus a zero-fill, so "missing" stays visible to the model instead of
silently looking like a real zero.


In [3]:
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
LOG_SOURCE_FEATURES = ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

# Missingness check per the flyrank-data gotcha: does it follow content_type?
miss_by_type = df.groupby("content_type")[["word_count", "search_volume"]].apply(
    lambda g: g.isna().mean() * 100
)
print("Missingness (%) of word_count / search_volume, by content_type:")
print(miss_by_type.round(1))

feat = pd.DataFrame(index=df.index)
for col in NUMERIC_FEATURES:
    feat[f"has_{col}"] = df[col].notna().astype(int)
    feat[col] = df[col].fillna(0)
for col in LOG_SOURCE_FEATURES:
    feat[f"log_{col}"] = np.log1p(df[col].fillna(0))
cat_encoded = pd.get_dummies(df[CATEGORICAL_FEATURES].astype("category"), dummy_na=True)
feat = pd.concat([feat, cat_encoded], axis=1)

print(f"\nFinal feature matrix: {feat.shape[0]:,} rows x {feat.shape[1]} columns")
print(f"has_word_count=0 rows (missing, now flagged not zero-filled silently): "
      f"{(feat['has_word_count'] == 0).sum():,}")


Missingness (%) of word_count / search_volume, by content_type:
                    word_count  search_volume
content_type                                 
comparison article         0.0            0.0
feedly article             0.0          100.0
keyword article           28.3            1.4

Final feature matrix: 30,000 rows x 71 columns
has_word_count=0 rows (missing, now flagged not zero-filled silently): 7,699


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing? | Type | Available before prediction moment? |
|---|---|---|---|---|
| `ctr`, `avg_position` | current search performance | `avg_position=0` means no ranking data (not rank zero) — kept as-is, model can learn "0 = no signal" | numeric | Yes — snapshot-level, not derived from the future |
| `impressions_90d`, `clicks_90d`, `sessions_90d` (+ log) | trailing 90-day traffic volume | rare | numeric | **Conditionally** — see leakage hunt below, these numerically overlap the label's own sub-windows |
| `word_count`, `char_count` | on-page size | missing concentrated in specific `content_type`s (see printed table) | numeric, `has_*` flag added | Yes |
| `days_since_last_update`, `content_age_days` | staleness / age | rare | numeric | Yes |
| `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | behavioral signals; can exceed 100 (different measurement systems, not a bug) | rare | numeric | Yes |
| `content_type`, `main_intent`, `*_tier` columns | categorical context | `dummy_na=True` keeps missing as its own category rather than dropping rows | categorical | Yes |
| `content_id`, `client_id` | pseudonymous identifiers | — | — | Grouping/joining only — **never a feature** |
| `trend_direction`, `trend_pct` | the label's own source | — | — | **Never a feature — this IS the label** |


## 3. The leakage hunt

Running the attack checklist from `hunting-leakage-and-validating`:

**Category 1 — label-derived features.** `is_declining_label = (trend_direction == "down")`.
`trend_direction` is computed from `trend_pct`. Test: is either column perfectly separable from
the label? Yes, by construction — confirmed below and excluded outright.

**Category 2 — future/overlapping windows.** `impressions_90d` (and `clicks_90d`, `sessions_90d`)
claim to be trailing-90-day aggregates, but the data also carries `impressions_last_30d` and
`impressions_prev_30d`. If the 90-day figure is built from sub-windows that also feed
`trend_pct` (the label's source), that's a feature window overlapping the label window — exactly
the category-2 leak this skill warns about. Tested directly below with real numbers, not assumed.

**Category 3 — decision-derived / product flags.** This starter file carries no existing
system score or human-decision flag column (no `priority_flag`, `current_score`, etc. among its
44 columns) — category 3 does not apply here.


In [4]:
# Category 1: confirm trend_direction/trend_pct are label-derived, not merely correlated.
print("trend_direction vs is_declining_label (should be a perfect 1:1 map):")
print(pd.crosstab(df["trend_direction"], df["is_declining_label"]))
print()

# Category 2: does impressions_90d numerically overlap the last-30d / prev-30d sub-windows
# that trend_pct (the label's source) is computed from?
sub_window_sum = df["impressions_last_30d"].fillna(0) + df["impressions_prev_30d"].fillna(0)
corr = df["impressions_90d"].corr(sub_window_sum)
ratio = (sub_window_sum / df["impressions_90d"].replace(0, np.nan)).median()
print(f"Correlation(impressions_90d, impressions_last_30d + impressions_prev_30d) = {corr:.4f}")
print(f"Median ratio of (last_30d + prev_30d) / 90d total = {ratio:.3f}")
print("-> Near-1.0 correlation confirms impressions_90d overlaps the exact sub-windows")
print("   trend_pct is computed from. Same check applies to clicks_90d / sessions_90d by")
print("   construction (same 90d/30d/prev-30d family of columns).")
print()
print("VERDICT: category-2 leakage risk CONFIRMED for impressions_90d, clicks_90d, sessions_90d")
print("(and their log transforms). Flagged as HIGH-RISK, carried into Week 5 modeling as a")
print("monitored risk rather than dropped immediately -- the actual cost (how much precision")
print("depends on them) is only known once tested with an ablation, which is exactly what")
print("ML-09 (work/notebooks/w06_validation_audit.ipynb) does: a real with/without comparison")
print("showing precision@50 drops from 0.74 to 0.46 when these features are removed. ML-10")
print("(w07_action_playbook.ipynb) acts on that finding and ships the leakage-clean version.")


trend_direction vs is_declining_label (should be a perfect 1:1 map):
is_declining_label     0      1
trend_direction                
down                   0  16262
flat                1152      0
new                 2236      0
stable              5962      0
up                  4388      0

Correlation(impressions_90d, impressions_last_30d + impressions_prev_30d) = 0.9805
Median ratio of (last_30d + prev_30d) / 90d total = 0.570
-> Near-1.0 correlation confirms impressions_90d overlaps the exact sub-windows
   trend_pct is computed from. Same check applies to clicks_90d / sessions_90d by
   construction (same 90d/30d/prev-30d family of columns).

VERDICT: category-2 leakage risk CONFIRMED for impressions_90d, clicks_90d, sessions_90d
(and their log transforms). Flagged as HIGH-RISK, carried into Week 5 modeling as a
monitored risk rather than dropped immediately -- the actual cost (how much precision
depends on them) is only known once tested with an ablation, which is exactly what
M

## 4. What I excluded and why

- **`trend_direction`, `trend_pct`** — excluded outright. These define the label; including them
  means the model learns the label from itself (category 1, confirmed above by the perfect
  crosstab).
- **`content_id`, `client_id`** — excluded from features. Pseudonymous identifiers; used only for
  the client-grouped train/test split (Week 5+), never as model inputs.
- **`impressions_90d`, `clicks_90d`, `sessions_90d` (+ log transforms)** — flagged as HIGH-RISK
  category-2 leakage (numeric overlap with the label's own sub-windows, confirmed above).
  Carried forward as a *monitored* risk into the Week 5 baseline model, then formally tested with
  an ablation in ML-09 and dropped from the production model in ML-10 once the real cost
  (precision@50: 0.74 → 0.46) was measured — disclosed, not silently patched.
- **No product-flag columns to exclude** — none exist in this starter file (category 3 doesn't
  apply here).


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (verified: all code cells extracted and run
      as a script, exit code 0)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
